# Variant Analysis
Per-run variant deep-dives, cross-sample comparisons, overlap heatmaps, and same-donor similarity.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import scipy.stats as stats

DB_PATH = 'results/variants.db'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")
runs['label'] = runs['cell_line'].fillna(runs['sample'])
print(f'Loaded {len(runs)} runs.')

## Per-run deep-dive
Select a run to see depth distribution, quality score distribution, allele frequency spectrum, variant types, and chromosomal distribution.

In [ ]:
run_labels = [f"{r['run_id']} ({r['label']})" for _, r in runs.iterrows()]
run_select = widgets.Dropdown(
    options=list(zip(run_labels, runs['run_id'])),
    description='Run:',
    layout=widgets.Layout(width='60%')
)
out_run = widgets.Output()

def show_run_detail(run_id):
    geno = query("""
        SELECT v.chromosome AS chrom, v.position AS pos, v.ref_allele AS ref,
               v.alt_allele AS alt, g.depth, g.quality AS qual, g.allele_freq,
               g.filter_status, v.variant_type
        FROM genotype_calls g
        JOIN runs r ON g.run_id = r.run_id
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE r.run_id = ? AND g.filter_status = 'PASS'
        LIMIT 500000
    """, params=(run_id,))
    if geno.empty:
        print(f'No passing variants for {run_id}.')
        return

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            'Depth distribution', 'QUAL score distribution', 'Allele frequency spectrum',
            'Variant types', 'Chromosomal distribution', 'Depth vs AF'
        ),
        horizontal_spacing=0.1, vertical_spacing=0.15
    )
    fig.add_trace(go.Histogram(x=geno['depth'], nbinsx=60, marker_color='#457b9d', name='Depth'), row=1, col=1)
    fig.add_trace(go.Histogram(x=geno['qual'],  nbinsx=60, marker_color='#2d6a4f', name='QUAL'),  row=1, col=2)
    fig.add_trace(go.Histogram(x=geno['allele_freq'], nbinsx=50, marker_color='#e9c46a', name='AF'), row=1, col=3)

    if 'variant_type' in geno.columns and geno['variant_type'].notna().any():
        vt = geno['variant_type'].value_counts()
        fig.add_trace(go.Bar(x=vt.index, y=vt.values, marker_color='#e76f51', name='Var types'), row=2, col=1)

    chrom_order = [str(c) for c in range(1, 23)] + ['X', 'Y', 'MT']
    chrom_counts = geno['chrom'].str.replace('chr', '', regex=False).value_counts().reindex(chrom_order).dropna()
    fig.add_trace(go.Bar(x=chrom_counts.index, y=chrom_counts.values, marker_color='#264653', name='Chrom'), row=2, col=2)

    sample_frac = geno.sample(min(5000, len(geno)))
    fig.add_trace(go.Scatter(
        x=sample_frac['depth'], y=sample_frac['allele_freq'],
        mode='markers', marker=dict(size=3, color='#a8dadc', opacity=0.5), name='Depth/AF'
    ), row=2, col=3)

    row_info = runs[runs['run_id'] == run_id].iloc[0]
    fig.update_layout(
        height=600,
        title=f"{run_id}  ({row_info['label']})  —  {len(geno):,} passing variants",
        showlegend=False
    )
    fig.show()

def on_run_change(change):
    with out_run:
        clear_output(wait=True)
        show_run_detail(change['new'])

run_select.observe(on_run_change, names='value')
display(run_select, out_run)
if runs.shape[0] > 0:
    show_run_detail(runs['run_id'].iloc[0])

## Cross-sample comparison
Shared variants, depth, and AF comparison between two selected runs.

In [ ]:
select_a = widgets.Dropdown(options=list(zip(run_labels, runs['run_id'])), description='Run A:', layout=widgets.Layout(width='55%'))
select_b = widgets.Dropdown(options=list(zip(run_labels, runs['run_id'])), description='Run B:', layout=widgets.Layout(width='55%'))
if len(runs) > 1:
    select_b.value = runs['run_id'].iloc[1]
out_compare = widgets.Output()

def compare_runs(run_a, run_b):
    if run_a == run_b:
        print('Select two different runs.')
        return
    geno_a = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key,
               g.depth, g.allele_freq
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (run_a,))
    geno_b = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key,
               g.depth, g.allele_freq
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (run_b,))
    if geno_a.empty or geno_b.empty:
        print('No passing variants for one or both runs.')
        return

    set_a, set_b = set(geno_a['key']), set(geno_b['key'])
    overlap = set_a & set_b
    jaccard = len(overlap) / len(set_a | set_b) if (set_a | set_b) else 0
    cont_ab = len(overlap) / len(set_a) if set_a else 0
    cont_ba = len(overlap) / len(set_b) if set_b else 0

    print(f'Variants in A: {len(set_a):,}  |  B: {len(set_b):,}  |  shared: {len(overlap):,}')
    print(f'Jaccard: {jaccard:.4f}  |  A⊂B: {cont_ab:.4f}  |  B⊂A: {cont_ba:.4f}')

    shared_keys = list(overlap)
    df_a = geno_a[geno_a['key'].isin(shared_keys)].set_index('key').add_suffix('_a')
    df_b = geno_b[geno_b['key'].isin(shared_keys)].set_index('key').add_suffix('_b')
    merged = df_a.join(df_b).dropna()

    label_a = runs[runs['run_id'] == run_a]['label'].iloc[0]
    label_b = runs[runs['run_id'] == run_b]['label'].iloc[0]

    fig = make_subplots(rows=1, cols=2, subplot_titles=('Depth at shared sites', 'Allele frequency at shared sites'))
    if not merged.empty:
        sample_m = merged.sample(min(5000, len(merged)))
        fig.add_trace(go.Scatter(
            x=sample_m['depth_a'], y=sample_m['depth_b'],
            mode='markers', marker=dict(size=3, color='#264653', opacity=0.4),
            name='depth'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=sample_m['allele_freq_a'], y=sample_m['allele_freq_b'],
            mode='markers', marker=dict(size=3, color='#e76f51', opacity=0.4),
            name='AF'
        ), row=1, col=2)
    fig.update_xaxes(title_text=label_a, row=1, col=1)
    fig.update_yaxes(title_text=label_b, row=1, col=1)
    fig.update_xaxes(title_text=label_a, row=1, col=2)
    fig.update_yaxes(title_text=label_b, row=1, col=2)
    fig.update_layout(height=420, title=f'Shared sites: {label_a} vs {label_b}', showlegend=False)
    fig.show()

def on_compare_change(_):
    with out_compare:
        clear_output(wait=True)
        compare_runs(select_a.value, select_b.value)

select_a.observe(on_compare_change, names='value')
select_b.observe(on_compare_change, names='value')
display(select_a, select_b, out_compare)
if len(runs) >= 2:
    compare_runs(runs['run_id'].iloc[0], runs['run_id'].iloc[1])

## Variant overlap heatmap
All-vs-all Jaccard similarity and containment matrix for all runs.

In [ ]:
print('Loading variant keys for all runs (may take a moment)...')
all_keys = {}
for run_id in runs['run_id']:
    df = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (run_id,))
    all_keys[run_id] = set(df['key'].tolist()) if not df.empty else set()

run_ids = runs['run_id'].tolist()
labels  = runs['label'].tolist()
n = len(run_ids)
jaccard_mat = np.zeros((n, n))
contain_mat = np.zeros((n, n))
for i, ri in enumerate(run_ids):
    for j, rj in enumerate(run_ids):
        si, sj = all_keys[ri], all_keys[rj]
        union = len(si | sj)
        inter = len(si & sj)
        jaccard_mat[i, j] = inter / union if union else 0
        contain_mat[i, j] = inter / len(si) if si else 0

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Jaccard similarity (all-vs-all)', 'Containment (row ⊂ col)'),
    horizontal_spacing=0.12
)
for mat, col, cscale in [(jaccard_mat, 1, 'Blues'), (contain_mat, 2, 'Greens')]:
    fig.add_trace(go.Heatmap(
        z=mat, x=labels, y=labels,
        colorscale=cscale, zmin=0, zmax=1,
        text=np.round(mat, 3),
        texttemplate='%{text}', showscale=True
    ), row=1, col=col)

fig.update_layout(height=500, title='Variant overlap across runs')
fig.update_xaxes(tickangle=45)
fig.show()

jaccard_df = pd.DataFrame(jaccard_mat, index=labels, columns=labels)
upper = jaccard_df.where(np.triu(np.ones(jaccard_df.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print('Top 10 most similar run pairs (Jaccard):')
display(upper.head(10).reset_index().rename(columns={0:'jaccard', 'level_0':'Run A', 'level_1':'Run B'}))

## Similarity distributions: same-donor vs different-donor
Tests whether runs from the same cell line cluster together by variant overlap.

In [ ]:
same_donor_scores, diff_donor_scores = [], []
for i, ri in enumerate(run_ids):
    li = runs[runs['run_id'] == ri]['cell_line'].iloc[0]
    for j, rj in enumerate(run_ids):
        if j >= i:
            continue
        lj = runs[runs['run_id'] == rj]['cell_line'].iloc[0]
        score = jaccard_mat[i, j]
        if pd.notna(li) and pd.notna(lj) and li == lj:
            same_donor_scores.append(score)
        else:
            diff_donor_scores.append(score)

if same_donor_scores and diff_donor_scores:
    stat, pval = stats.mannwhitneyu(same_donor_scores, diff_donor_scores, alternative='greater')
    print(f'Same-donor: {len(same_donor_scores)} pairs  |  median = {np.median(same_donor_scores):.4f}')
    print(f'Diff-donor: {len(diff_donor_scores)} pairs  |  median = {np.median(diff_donor_scores):.4f}')
    print(f'Mann-Whitney U = {stat:.0f}, p = {pval:.4e}')

    fig = go.Figure()
    fig.add_trace(go.Violin(y=same_donor_scores, name='Same donor', box_visible=True, meanline_visible=True, fillcolor='#2d6a4f', line_color='#2d6a4f'))
    fig.add_trace(go.Violin(y=diff_donor_scores, name='Diff donor', box_visible=True, meanline_visible=True, fillcolor='#e76f51', line_color='#e76f51'))
    fig.update_layout(height=400, title=f'Variant overlap similarity: same- vs diff-donor (p={pval:.3e})', yaxis_title='Jaccard similarity')
    fig.show()
else:
    print('Not enough same-donor pairs to compare (need runs from at least 2 runs of the same cell_line).')
    if diff_donor_scores:
        print(f'Diff-donor pairs: {len(diff_donor_scores)}  |  median = {np.median(diff_donor_scores):.4f}')

## Raw SQL explorer

In [ ]:
sql_box = widgets.Textarea(
    value="SELECT v.chromosome, COUNT(*) AS n\nFROM genotype_calls g\nJOIN variants v ON g.variant_id = v.variant_id\nWHERE g.run_id = (SELECT run_id FROM runs LIMIT 1) AND g.filter_status='PASS'\nGROUP BY v.chromosome\nORDER BY n DESC;",
    layout=widgets.Layout(width='100%', height='100px')
)
run_btn = widgets.Button(description='Run query', button_style='primary')

def on_run_query(btn):
    clear_output(wait=False)
    display(sql_box, run_btn)
    try:
        display(query(sql_box.value))
    except Exception as e:
        print(f'Error: {e}')

run_btn.on_click(on_run_query)
display(sql_box, run_btn)